In [ ]:
# %pip 매직 커맨드를 사용하여 현재 Jupyter 커널의 Python 환경(.venv)에 패키지를 설치합니다.
%pip install "google-cloud-aiplatform[adk,agent_engines,evaluation]" pandas

# Agent 개발

In [ ]:
import os
from dotenv import load_dotenv
from agentplatform.frameworks import AdkApp
from google.adk.agents import Agent

# 루트 .env 파일 로드
load_dotenv(override=True)
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-3.8-flash")

# 1. 에이전트가 사용할 도구(Tools) 함수 정의
def search_products(query: str):
    """Searches for products based on a query."""
    if "headphones" in query.lower():
        return {"products": [{"name": "Wireless Headphones", "id": "B08H8H8H8H"}]}
    else:
        return {"products": []}

def get_product_details(product_id: str):
    """Gets the details for a given product ID."""
    if product_id == "B08H8H8H8H":
        return {"details": "Noise-cancelling, 20-hour battery life."}
    else:
        return {"error": "Product not found."}

def add_to_cart(product_id: str, quantity: int):
    """Adds a specified quantity of a product to the cart."""
    return {"status": f"Added {quantity} of {product_id} to cart."}

# 2. Local Agent (agent) 객체 생성
agent = Agent(
    model=GEMINI_MODEL,
    name="ecommerce_agent",
    instruction="You are an ecommerce expert",
    tools=[search_products, get_product_details, add_to_cart],
)

app = AdkApp(agent=agent)

# 환경 변수 설정 및 Client 초기화

In [ ]:
import os
import agentplatform
from dotenv import load_dotenv

# 루트 .env 파일 로드
load_dotenv(override=True)

PROJECT_ID = os.getenv("GOOGLE_CLOUD_PROJECT")
LOCATION = os.getenv("GCP_RESOURCES_LOCATION", "us-central1")
# 결과 저장용 GCS 경로 설정
GCS_DEST = os.getenv("STAGING_BUCKET_URI")

print(f"Project: {PROJECT_ID}, Eval Location: {LOCATION}, GCS_DEST: {GCS_DEST}")

# 2. Agent Platform Client 초기화
client = agentplatform.Client(
    project=PROJECT_ID,
    location=LOCATION,
)

# 평가용 데이터셋 (evaluation dataset) 구성

In [ ]:
import pandas as pd
from agentplatform import types

pd.set_option("display.max_colwidth", None)

# 세션 초기 상태 및 
session_inputs = types.evals.SessionInput(
    user_id="user_123",
    state={},
)
# 평가 프롬프트 정의
agent_prompts = [
    "Search for 'noise-cancelling headphones'.",
    "Show me the details for product 'B08H8H8H8H'.",
    "Add one pair of 'B08H8H8H8H' to my shopping cart.",
    "Find 'wireless earbuds' and then add the first result to my cart.",
]
# 위 정보를 가지고 Pandas DataFrame 형태의 데이터셋
agent_dataset = pd.DataFrame({
    "prompt": agent_prompts,
    "session_inputs": [session_inputs] * len(agent_prompts),
})

display(agent_dataset)

# Local Agent 대상으로 평가 데이터 셋을 가지고 추론 실행 (프롬프트, 응답 결과, 도구 호출 체인 값 포함)

In [ ]:
import json

# 평가 데이터 셋을 가지고 추론 실행 (최종 답변 생성)
# 도구 호출 인자, 도구의 반환값, 모델의 생성 텍스트, reasoning trace 등이 모두 캡처된 결과 데이터셋 객체가 반환됩니다.
dataset_with_inference = client.evals.run_inference(
    agent=agent,  # 로컬 메모리에 있는 ADK Agent 객체 전달
    src=agent_dataset,
)

# 추론 실행 결과의 출력
df = dataset_with_inference.eval_dataset_df.copy()
df["tools_called"] = df["intermediate_events"].apply(
    lambda ev: [p["function_call"]["name"] for e in (json.loads(ev) if isinstance(ev, str) else ev or []) for p in e.get("content", {}).get("parts", []) if "function_call" in p]
)
display(df[["prompt", "tools_called", "response"]])

# 평가 수행 (Async 방식으로 평가 수행)

In [ ]:
# Local Agent 대상으로 에이전트 평가 실행 (Evaluation Run 생성)
eval_run = client.evals.create_evaluation_run(
    dataset=dataset_with_inference,  # 추론 결과답변)가 포함된 데이터셋 전달
    metrics=[
        types.RubricMetric.FINAL_RESPONSE_QUALITY,  # 최종 답변 품질
        types.RubricMetric.TOOL_USE_QUALITY,        # 도구 사용 적절성
        types.RubricMetric.HALLUCINATION,           # 환각 감지
        types.RubricMetric.SAFETY,                  # 안정성
    ], 
    dest=f"{GCS_DEST}/eval_results",
)

# 평가 완료 대기 (평가를 Async 방식으로 수행했기 때문에, 평가 완료 될때까지 상태체크하면서 대기)

In [ ]:
import time

# 평가가 끝날 때까지 15초 간격으로 상태를 계속 확인합니다.
while eval_run.state not in ("SUCCEEDED", "FAILED", "CANCELLED"):
    print(f"⏳ 채점 진행 중... 현재 상태: {eval_run.state} (30초 후 다시 확인)")
    time.sleep(30)
    # 서버에서 최신 상태를 새로고침(fetch)해옵니다.
    eval_run = client.evals.get_evaluation_run(name=eval_run.name)

print(f"🎉 평가 완료! 최종 상태: {eval_run.state}")

# 평가 리포트 출력

In [ ]:
eval_run = client.evals.get_evaluation_run(
    name=eval_run.name,
    include_evaluation_items=True,
)

#############################
# 평기 리포트 출력을 위한 헬퍼 함수
def display_eval_report(res):
    # 1. EvaluationRun 객체에서 세부 채점 결과 추출
    if hasattr(res, "evaluation_item_results") and res.evaluation_item_results:
        res = res.evaluation_item_results

    # 2. 요약 점수 출력
    display(
        pd.DataFrame([m.model_dump() for m in res.summary_metrics])[
            ["metric_name", "mean_score", "stdev_score"]
        ]
    )

    # 3. 문항별 점수 + 감점 사유 출력
    base_df = res.evaluation_dataset[0].eval_dataset_df
    
    # ⭐️ 'prompt', 'response' 컬럼 중 실제 존재하는 컬럼만 안전하게 추출
    available_cols = [c for c in ["prompt", "response"] if c in base_df.columns]
    if not available_cols:
        available_cols = [base_df.columns[0]]  # 아무것도 매칭 안 되면 첫 번째 컬럼 사용
        
    df = base_df[available_cols].copy()
    
    # 만약 response 컬럼이 없다면 안내 문구로 채움
    if "response" not in df.columns:
        df["response"] = "(No response column in dataset)"

    # 문항별 점수 및 감점 사유 매핑
    for i, c in enumerate(res.eval_case_results):
        m = c.response_candidate_results[0].metric_results
        for k, v in m.items():
            df.loc[i, k.replace("_v1", "")] = v.score
        df.loc[i, "fail_reasons"] = (
            "\n".join(
                [
                    f"• [{k.replace('_v1', '')}] {rv.reasoning}"
                    for k, v in m.items()
                    for rv in (v.rubric_verdicts or [])
                    if not rv.verdict
                ]
            )
            or "Pass ✅"
        )
    display(df)
#############################

# 평가 리포트 내용 출력
print(f"🎉 Evaluation Report")
display_eval_report(eval_run)